In [5]:
from openai import OpenAI
from langchain.document_loaders import PyPDFLoader
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm
import os 
import shutil
import openpyxl
from openpyxl.styles import PatternFill
import logging
import time
import re


# Coarse-grained outcome labels

In [ ]:
df = pd.read_excel("patient_data.xlsx", sheet_name="Sheet1")

LLM_MODEL = "gpt-5.1"
client = OpenAI(
    base_url="OPENAI_BASE_URL",
    api_key="OPENAI_API_KEY",                             
) 


def fetch_prompts_response(content, max_retries, sleep_duration=1):
    retries = 0
    while retries < max_retries:
        try:
            response = client.chat.completions.create(
                messages=[
                    {"role": "user", "content": content}
                ],
                temperature=0.1,
                model=LLM_MODEL,
            )
            return response.choices[0].message.content
        except Exception as e:
            logging.error(f"Error during API call: {str(e)}")
            retries += 1
            time.sleep(sleep_duration)
            logging.error(f"Retry {retries}/{max_retries}...")
            if retries == max_retries:
                raise Exception("Max retries reached, failing...")
    return None

def clean_json_codeblock(text):
    match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    if match:
        return match.group(1).strip()
    # If there is no code block, simply return the original text.
    return text.strip()

TREE_CATEGORIES_PROMPT = """
You are a clinical outcome annotation assistant.

Task:
Classify the following outcome text into one of these three labels:

- positive
- stable/neutral
- negative

Label definitions:
- positive: remission, complete resolution, regression, improvement, no recurrence, no metastasis, disease-free status, or other clearly favorable outcome.
- stable/neutral: stable disease, no significant change, unchanged condition, continued follow-up/observation, or no clearly reported treatment effect.
- negative: recurrence, progression, metastasis, death, worsening condition, treatment failure, uncontrolled disease, or other clearly unfavorable outcome.

Decision rules:
1. Assign exactly one label.
2. Use the overall final clinical outcome as the main basis for labeling.
3. If both positive and negative events are mentioned, prioritize the final overall outcome.
4. If the text does not clearly indicate improvement or worsening, classify as stable/neutral.

Output requirement:
Return only one label:
positive
stable/neutral
negative

Outcome text:
{outcome_text}
"""

In [7]:
df['outcome_coarse_label'] = ''

for idx in tqdm(range(len(df))):
    outcome_text = df.loc[idx, "outcome"]
    content = TREE_CATEGORIES_PROMPT.format(outcome_text=outcome_text)
    response = fetch_prompts_response(content, max_retries=3)
    cleaned_response = clean_json_codeblock(response)

    df.loc[idx, "outcome_coarse_label"] = cleaned_response

100%|██████████| 2134/2134 [1:27:23<00:00,  2.46s/it]  


In [ ]:
df.to_excel("patient_data_with_coarse_labels.xlsx", index=False)

# Fine-grained Outcome Labels

In [ ]:
df = pd.read_excel("patient_data.xlsx", sheet_name="Sheet1")

LLM_MODEL = "deepseek-chat"
client = OpenAI(
    base_url="OPENAI_BASE_URL",
    api_key="OPENAI_API_KEY",                             
) 


def fetch_prompts_response(content, max_retries, sleep_duration=1):
    retries = 0
    while retries < max_retries:
        try:
            response = client.chat.completions.create(
                messages=[
                    {"role": "user", "content": content}
                ],
                temperature=0.1,
                model=LLM_MODEL,
            )
            return response.choices[0].message.content
        except Exception as e:
            logging.error(f"Error during API call: {str(e)}")
            retries += 1
            time.sleep(sleep_duration)
            logging.error(f"Retry {retries}/{max_retries}...")
            if retries == max_retries:
                raise Exception("Max retries reached, failing...")
    return None

def clean_json_codeblock(text):
    match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    if match:
        return match.group(1).strip()
    # If there is no code block, simply return the original text.
    return text.strip()

FINE_PROMPT = """
You are a medical information extraction assistant.

Your task is to read one free-text outcome statement and assign 6 auxiliary labels.

Use only the information explicitly stated in the text. Do not infer unstated facts using outside medical knowledge. If the text does not provide enough evidence for a label, output "unknown". If both early improvement and later worsening are mentioned, prioritize the final explicit outcome or the more severe outcome. Judge each label independently.

Assign the following labels:

1. survival_label
Allowed values:
- alive
- dead
- unknown

Rules:
- Assign "alive" if the text explicitly states alive, survived, doing well, remains well, still alive, or similar.
- Assign "dead" if the text explicitly states died, dead, death, expired, passed away, or similar.
- Otherwise assign "unknown".

2. metastasis_label
Allowed values:
- no_metastasis
- metastasis
- unknown

Rules:
- Assign "no_metastasis" if the text explicitly states no metastasis, no metastatic disease, metastasis-free, no distant spread, or similar.
- Assign "metastasis" if the text explicitly states metastasis, metastases, metastatic disease, liver metastasis, bone metastasis, distant spread, or similar.
- Otherwise assign "unknown".

3. recurrence_label
Allowed values:
- no_recurrence
- recurrence
- unknown

Rules:
- Assign "no_recurrence" if the text explicitly states no recurrence, recurrence-free, no local recurrence, no ocular relapse, or similar.
- Assign "recurrence" if the text explicitly states recurrence, recurred, relapse, tumor breakthrough, treated for recurrence, or similar.
- Otherwise assign "unknown".

4. adverse_event_label
Allowed values:
- yes
- no
- unknown

Rules:
- Assign "yes" if the text explicitly mentions treatment-related adverse events, complications, toxicity, cataract, glaucoma, radiation retinopathy, corneal toxicity, hepatotoxicity, colitis, hypothyroidism, neutropenia, or similar.
- Assign "no" if the text explicitly states no adverse events, no complications, tolerated well, without complications, or similar.
- Otherwise assign "unknown".

Important:
- Only label "no" when the absence of adverse events or complications is explicitly stated.
- If adverse effects or complications occurred at any point, even if later resolved, assign "yes".

5. vision_label
Allowed values:
- improved
- stable
- worsened
- unknown

Rules:
- Assign "improved" if the text explicitly states visual acuity improved, vision improved, BCVA improved, recovered vision, subjective improvement in vision, or similar.
- Assign "stable" if the text explicitly states vision stable, visual acuity stable, maintained vision, vision preserved, no loss of vision, or similar.
- Assign "worsened" if the text explicitly states visual acuity decreased, vision worsened, declined to light perception, blind, no light perception, vision loss, or similar.
- Otherwise assign "unknown".

Important:
- If both improvement and worsening are mentioned, prioritize the final explicit visual outcome or the more severe visual outcome.

6. eye_preservation_label
Allowed values:
- Eye preserved
- Eye enucleated
- unknown

Rules:
- Assign "Eye preserved" if the text explicitly states globe salvage, eye preserved, both eyes salvaged, globe maintained, or similar.
- Assign "Eye enucleated" if the text explicitly states enucleation, eye enucleated, underwent enucleation, or similar.
- Otherwise assign "unknown".

Important:
- If the eye was ultimately enucleated, prioritize "Eye enucleated".
- If only exenteration is mentioned, assign "unknown".


General principles:
- Judge each label independently.
- Do not infer no_recurrence from alive.
- Do not infer no_metastasis from no recurrence.
- Do not infer tumor controlled from vision improvement alone.
- Prefer explicit statements over assumptions.

Return only one JSON object with exactly these 6 fields and no extra text:

{{
  "survival_label": "",
  "metastasis_label": "",
  "recurrence_label": "",
  "adverse_event_label": "",
  "vision_label": "",
  "eye_preservation_label": ""
}}

Outcome text:
{outcome_text}
"""

In [ ]:
df['survival_label'] = ''
df['metastasis_label'] = ''
df['recurrence_label'] = ''
df['adverse_event_label'] = ''
df['vision_label'] = ''
df['eye_preservation_label'] = ''

In [ ]:
for idx in tqdm(range(len(df))):
    outcome_text = df.loc[idx, "outcome"]
    content = FINE_PROMPT.format(outcome_text=outcome_text)
    response_json = fetch_prompts_response(content, max_retries=3)
    response = json.loads(clean_json_codeblock(response_json))
    df.loc[idx, "survival_label"] = response.get("survival_label", "unknown")
    df.loc[idx, "metastasis_label"] = response.get("metastasis_label", "unknown")
    df.loc[idx, "recurrence_label"] = response.get("recurrence_label", "unknown")
    df.loc[idx, "adverse_event_label"] = response.get("adverse_event_label", "unknown")
    df.loc[idx, "vision_label"] = response.get("vision_label", "unknown")
    df.loc[idx, "eye_preservation_label"] = response.get("eye_preservation_label", "unknown")

In [ ]:
cols = [
    'survival_label',
    'metastasis_label',
    'recurrence_label',
    'adverse_event_label',
    'vision_label',
    'eye_preservation_label'
]
df[cols] = df[cols].replace('unknown', '')

df.to_excel("patient_data_with_fine_labels.xlsx", index=False)